<a href="https://colab.research.google.com/github/amakalarry/Synthetic-Crime-Scene-Research/blob/main/Augment%2BTrain_Val_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Augment + train with Albumentations (YOLO bbox-safe)

In [ ]:
#Importing Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/AllNewDataset_YOLO/images/train | head


ai_video01_frame_0.jpg
ai_video01_frame_120.jpg
ai_video01_frame_168.jpg
ai_video01_frame_24.jpg
ai_video01_frame_48.jpg
ai_video01_frame_96.jpg
ai_video02_frame_0.jpg
ai_video02_frame_48.jpg
ai_video02_frame_72.jpg
ai_video02_frame_96.jpg


Checking Label MAtching

In [ ]:
import os, glob

img_dir = "/content/drive/MyDrive/AllNewDataset_YOLO/images/train"
lbl_dir = "/content/drive/MyDrive/AllNewDataset_YOLO/labels/train"

imgs = sorted(glob.glob(img_dir + "/*.jpg"))
missing = []

for p in imgs[:200]:  # check first 200
    stem = os.path.splitext(os.path.basename(p))[0]
    if not os.path.exists(os.path.join(lbl_dir, stem + ".txt")):
        missing.append(stem)

print("Checked:", min(200, len(imgs)))
print("Missing labels:", len(missing))
print("Examples:", missing[:10])


Checked: 200
Missing labels: 0
Examples: []


Creating the dataset.yaml file

In [ ]:
yaml_text = """\
path: /content/drive/MyDrive/AllNewDataset_YOLO
train: images/train
val: images/val
test: images/test

names:
  0: Body
  1: Laptop
  2: Mobilephone
  3: GlassCup
  4: Keyboard
  5: Camera
  6: Knife
  7: Router
  8: Tablet
  9: Smartwatch
  10: TV
  11: flashdrive
  12: Smartspeaker
"""

out = "/content/drive/MyDrive/AllNewDataset_YOLO/dataset.yaml"
with open(out, "w") as f:
    f.write(yaml_text)

print("Wrote:", out)


Wrote: /content/drive/MyDrive/AllNewDataset_YOLO/dataset.yaml


In [ ]:
#Removing duplicate lines in each .txt file for the labels

import os
from pathlib import Path

ROOT = "/content/drive/MyDrive/AllNewDataset_YOLO/labels"
splits = ["train", "val", "test"]

total_files = 0
changed_files = 0
total_removed = 0

for sp in splits:
    d = Path(ROOT) / sp
    if not d.exists():
        print("Missing:", d)
        continue

    for p in d.glob("*.txt"):
        total_files += 1
        lines = p.read_text().strip().splitlines()
        if not lines:
            continue

        # preserve order while removing duplicates
        seen = set()
        new_lines = []
        removed = 0
        for line in lines:
            line = line.strip()
            if not line:
                continue
            if line in seen:
                removed += 1
            else:
                seen.add(line)
                new_lines.append(line)

        if removed > 0:
            p.write_text("\n".join(new_lines) + "\n")
            changed_files += 1
            total_removed += removed

print("✅ Done.")
print("Total label files checked:", total_files)
print("Files changed:", changed_files)
print("Total duplicate lines removed:", total_removed)


✅ Done.
Total label files checked: 1303
Files changed: 8
Total duplicate lines removed: 119


In [ ]:
!rm -f /content/drive/MyDrive/AllNewDataset_YOLO/labels/train.cache
!rm -f /content/drive/MyDrive/AllNewDataset_YOLO/labels/val.cache
!rm -f /content/drive/MyDrive/AllNewDataset_YOLO/labels/test.cache


Train (YOLOv11s) with tuned augmentation + better hyperparams

In [ ]:
!pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.0 MB/s eta 0:00:00


In [ ]:
!mkdir -p /content/drive/MyDrive/AllNewDataset_YOLO/runs

In [ ]:
!which yolo


/usr/local/bin/yolo


#Training

In [ ]:
!nohup yolo detect train \
  model="/content/drive/MyDrive/AllNewDataset_YOLO/yolo11s.pt" \
  data="/content/drive/MyDrive/AllNewDataset_YOLO/dataset.yaml" \
  imgsz=640 \
  epochs=300 \
  batch=8 \
  workers=8 \
  cache=disk \
  seed=42 \
  augment=True \
  optimizer=AdamW \
  cos_lr=True \
  erasing=0.2 \
  lr0=0.002 lrf=0.01 weight_decay=0.0005 warmup_epochs=3 \
  patience=40 \
  hsv_h=0.015 hsv_s=0.70 hsv_v=0.40 \
  degrees=7 translate=0.05 scale=0.10 shear=3 perspective=0.0 \
  fliplr=0.5 flipud=0.0 \
  mosaic=1.0 mixup=0.05 copy_paste=0.05 \
  project="/content/drive/MyDrive/AllNewDataset_YOLO/runs" \
  name="/content/drive/MyDrive/AllNewDataset_YOLO/runs/train_yolo11s_x3" \
  save_period=10 \
  > "/content/drive/MyDrive/AllNewDataset_YOLO/runs/train_yolo11s_x3.log" 2>&1 &

#Tracking the process of the model training

In [ ]:
#Monitoring the progressing
!tail -f "/content/drive/MyDrive/AllNewDataset_YOLO/runs/train_yolo11s_x3.log"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.252 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.05, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/AllNewDataset_YOLO/dataset.yaml, degrees=7, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=F

Validation Phase

#Running Validation again for verification

In [ ]:
!yolo detect val \
  model="/content/drive/MyDrive/AllNewDataset_YOLO/runs/train_yolo11s_x3/weights/best.pt" \
  data="/content/drive/MyDrive/AllNewDataset_YOLO/dataset.yaml" \
  source="/content/drive/MyDrive/AllNewDataset_YOLO/images/val" \
  split=test \
  imgsz=640 \
  save=True \
save_txt=True \
save_conf=True \
project= /content/drive/MyDrive/AllNewDataset_YOLO/runs name=val_yolo11s\
> /content/drive/MyDrive/AllNewDataset_YOLO/runs/val_yolo11s.log  2>&1 &




#Testing using the Split Dataset

In [ ]:
!yolo detect predict \
  model="/content/drive/MyDrive/AllNewDataset_YOLO/runs/train_yolo11s_x3/weights/best.pt" \
  data="/content/drive/MyDrive/AllNewDataset_YOLO/dataset.yaml" \
  source="/content/drive/MyDrive/AllNewDataset_YOLO/images/test" \
  split=test \
  imgsz=640 \
  save=True \
save_txt=True \
save_conf=True \
project= /content/drive/MyDrive/AllNewDataset_YOLO/runs name=test_yolo11s\
> /content/drive/MyDrive/AllNewDataset_YOLO/runs/test_yolo11s.log  2>&1 &
